## Support Vector Machine 

In [ ]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
from geopandas import GeoDataFrame
from pyproj import Proj
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR
myProj = Proj("+proj=utm +zone=43 +north +datum=WGS84 +units=m +no_defs")   # Define Projection to be UTM 43N.

In [ ]:
df=pd.read_csv('Train_Test_REE_final.csv')
df.dropna(inplace=True)
df

In [ ]:
# Inverse projection to get Long, Lat from UTM coordinates. Then we define WGS 1984(also called EPSG:4326) coordinate reference system(crs)
long,lat=myProj(df.X.values,df.Y.values, inverse=True)
geometry = [Point(xy) for xy in zip(long, lat)]                  # All locations converted to point geometry
df = GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)        # Dataframe converted to Geospatial compatible dataframe(Geodataframe)
df

# Machine Learning Starts here

In [ ]:
# Split the dataframe into training (80%) and testing (20%) sets randomly
df_tr, df_ts = train_test_split(df, test_size=0.2, random_state=42)

# Column Normalization/Scaling


In [ ]:
X_tr = df_tr.iloc[:,3:-1].values
np.shape(X_tr)
X_tr = MinMaxScaler().fit_transform(X_tr)

In [ ]:
y_tr = df_tr['Prob'].values

In [ ]:
X_ts = df_ts.iloc[:,3:-1].values
np.shape(X_ts)
X_ts = MinMaxScaler().fit_transform(X_ts)
y_ts = df_ts['Prob'].values

# Choosing the best set of hyperparameters for optimum performance

In [ ]:
#Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import make_scorer, mean_absolute_error, explained_variance_score

#Define custom scorers (for evaluation)
scorers = {
    'MSE': 'neg_mean_squared_error',  # minimize MSE
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),  # minimize MAE
    'R2': 'r2',  # maximize R² score
    'Explained_Variance': make_scorer(explained_variance_score)  # maximize Explained Variance
}

#Define expanded parameter grid for SVR
param_grid = {
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],  # Try multiple kernels
     'C': np.logspace(-2, 2, 10),  # C from 0.01 to 1000 (log-spaced)
    #'C': np.logspace(-3, np.log10(100), 20),#C from 0.001 and 100
    'epsilon': np.linspace(0.01, 1, 10),  # Epsilon margin
    'gamma': ['scale', 'auto']  # Gamma parameter
}

#Perform hyperparameter tuning using RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=SVR(),
    param_distributions=param_grid,
    n_iter=100,             # 100 random combinations
    cv=5,                   # 5-fold cross-validation
    scoring=scorers,        # multiple scorers
    refit='R2',             # optimize based on R² score
    n_jobs=-1,              # use all CPU cores
    verbose=2,              # detailed output
    random_state=42         # fix randomness for reproducibility
)

# Fit the model
random_search.fit(X_tr, y_tr)

# Extract results from the tuning process
results = pd.DataFrame(random_search.cv_results_)

#  Convert negative scores to positive where needed
results['mean_test_MSE'] = -results['mean_test_MSE']
results['mean_test_MAE'] = -results['mean_test_MAE']


In [ ]:
# Train final SVR model with the best parameters
final_svr = SVR(**best_params)
final_svr.fit(X_tr, y_tr)

# Predict on training data
y_pred_tr = final_svr.predict(X_tr)

# Model performance on training data
print('Training data (Known to machine, human)')
mse = mean_squared_error(y_tr, y_pred_tr)
rmse = mse**0.5
print(f"Training MSE: {mse}")
print(f"Training RMSE: {rmse}")
print(f"Training MAE: {mean_absolute_error(y_tr, y_pred_tr)}")
print(f"Training R2 Score: {r2_score(y_tr, y_pred_tr)}")
print(f"Training Explained Variance: {explained_variance_score(y_tr, y_pred_tr)}")

# Predict on test data
y_pred_ts = final_svr.predict(X_ts)

# Model performance on test data
print('Test data (Unknown to machine, known to human)')
mse = mean_squared_error(y_ts, y_pred_ts)
rmse = mse**0.5
print(f"Test MSE: {mse}")
print(f"Test RMSE: {rmse}")
print(f"Test MAE: {mean_absolute_error(y_ts, y_pred_ts)}")
print(f"Test R2 Score: {r2_score(y_ts, y_pred_ts)}")
print(f"Test Explained Variance: {explained_variance_score(y_ts, y_pred_ts)}")


In [ ]:
X=df.iloc[:,3:-1].values
X = MinMaxScaler().fit_transform(X)
#y_ts = df_ts['Prob'].values
all_pred = final_svr.predict(X) 


In [ ]:
y=df['Prob'].values
print('Train+Test data Mean squared error=', mean_squared_error(y, all_pred))
df['Predicted_Probability']=all_pred

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance
result = permutation_importance(final_svr, X_tr, y_tr, n_repeats=10, random_state=42, scoring='r2')

# Feature importance scores
importance_scores = result.importances_mean

print("Permutation Feature Importances:")
for i, score in enumerate(importance_scores):
    print(f"Feature {i}: {score:.4f}")


In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance (for any kernel, including non-linear)
result = permutation_importance(final_svr, X_tr, y_tr, n_repeats=10, random_state=42, scoring='r2')

# Get feature importances from the result
feat_importances = pd.Series(result.importances_mean, index=df.iloc[:, 3:-2].columns)

# Plot the top 10 important features
feat_importances.nlargest(10).plot(kind='barh', figsize=(10, 6), color='skyblue')
plt.title("Top 10 Features by Permutation Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.grid(True)
plt.show()


In [ ]:
mdf=pd.read_csv('Master_REE.csv')
mdf.dropna(inplace=True)
mdf

In [ ]:
MX=mdf.iloc[:,2:].values
MX

In [ ]:
MX=mdf.iloc[:,2:].values
MX_scale = MinMaxScaler().fit_transform(MX)
all_pred = final_svr.predict(MX_scale) 
mdf['Predicted_Probability']=all_pred

In [ ]:
mdf.to_csv('REE_SVR.csv', index=False)